In [ ]:
#training
import torch
import torch.nn as nn
from torch.optim import AdamW
from transformers import (
    GPT2LMHeadModel,
    AutoTokenizer,
    get_cosine_schedule_with_warmup
)
from datasets import load_dataset
from torch.utils.data import DataLoader
from tqdm import tqdm
import math
from torch.cuda.amp import autocast, GradScaler
import os

# ---------------------------
# 1. Load and tokenize dataset
# ---------------------------
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:30%]")
tokenizer = AutoTokenizer.from_pretrained("gpt2-medium")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

block_size = 256

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=block_size)

dataset = dataset.train_test_split(test_size=0.1, seed=30)
train_dataset = dataset["train"].map(tokenize, batched=True)
val_dataset = dataset["test"].map(tokenize, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids"])
val_dataset.set_format(type="torch", columns=["input_ids"])

# ---------------------------
# 2. DataLoader
# ---------------------------
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

# ---------------------------
# 3. Load Pretrained GPT-2
# ---------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GPT2LMHeadModel.from_pretrained("gpt2-medium").to(device)
model.resize_token_embeddings(len(tokenizer))

# ---------------------------
# 4. Optimizer, Scheduler, AMP
# ---------------------------
optimizer = AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
total_steps = len(train_loader) * 5
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=100, num_training_steps=total_steps)
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
scaler = GradScaler()

# ---------------------------
# 5. Evaluation Function
# ---------------------------
def evaluate():
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            x = input_ids[:, :-1]
            y = input_ids[:, 1:]
            with autocast():
                outputs = model(x)
                logits = outputs.logits
                B, T, V = logits.shape
                loss = criterion(logits.reshape(B * T, V), y.reshape(B * T))
            val_loss += loss.item()
    model.train()
    return val_loss / len(val_loader)

# ---------------------------
# 6. Training Loop (with best model saving)
# ---------------------------
best_val_loss = float("inf")
save_path = "best_model_gpt2"

for epoch in range(5):
    total_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for i, batch in enumerate(loop):
        input_ids = batch["input_ids"].to(device)
        x = input_ids[:, :-1]
        y = input_ids[:, 1:]

        with autocast():
            outputs = model(x)
            logits = outputs.logits
            B, T, V = logits.shape
            loss = criterion(logits.reshape(B * T, V), y.reshape(B * T))

        if torch.isnan(loss):
            print(f"NaN loss at step {i}, skipping batch.")
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        scheduler.step()

        total_loss += loss.item()
        if i % 100 == 0:
            loop.set_postfix(loss=loss.item())

    avg_train_loss = total_loss / len(train_loader)
    val_loss = evaluate()
    perplexity = math.exp(val_loss)
    print(f"Epoch {epoch+1} complete. Train Loss: {avg_train_loss:.4f}, Val Loss: {val_loss:.4f}, Perplexity: {perplexity:.2f}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        model.save_pretrained(save_path)
        tokenizer.save_pretrained(save_path)
        print(f"Best model saved (Val Loss: {val_loss:.4f})")

In [8]:
#Text Generation
from transformers import GPT2LMHeadModel, AutoTokenizer
import torch

# Load the fine‑tuned model
model = GPT2LMHeadModel.from_pretrained("best_model_gpt2").to("cuda")
tokenizer = AutoTokenizer.from_pretrained("best_model_gpt2")

def generate_text(prompt: str, max_length=100, temperature=0.8, top_k=50, top_p=0.95):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")
    output = model.generate(
        input_ids,
        max_length=max_length,
        do_sample=True,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Example
prompt = "The future of technology will."
generated = generate_text(prompt)
print(generated)


The future of technology will. most likely. destroy us . 
  
 
 
 
  
  
 
 
  
 
  
 
 
 
 
  

  

  

  

 
 

 
 
 

 

 

 

 

 

 
 

 
 


In [7]:
import torch
import math
from transformers import GPT2LMHeadModel, AutoTokenizer
from torch.nn import CrossEntropyLoss

# Load your fine-tuned model
model = GPT2LMHeadModel.from_pretrained("best_model_gpt2").to("cuda")
tokenizer = AutoTokenizer.from_pretrained("best_model_gpt2")

criterion = CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

def compute_perplexity(text, block_size=256):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=block_size)
    input_ids = inputs.input_ids.to("cuda")

    with torch.no_grad():
        outputs = model(input_ids[:, :-1])
        logits = outputs.logits
        B, T, V = logits.shape
        loss = criterion(logits.reshape(B * T, V), input_ids[:, 1:].reshape(B * T))
    
    model.train()
    return math.exp(loss.item())

# Example
sample_text = "I hope you all liked our presentation."
perplexity = compute_perplexity(sample_text)
print(f"Perplexity for sample: {perplexity:.2f}")

Perplexity for sample: 133.37
